# 07_Temperature_Sweep — 溫度參數掃描（資金集中度優化）

## 目的

Phase 3-D 核心研究：尋找最佳的 `WEIGHT_TEMP` ( Softmax 溫度參數 )。

Softmax 決定了我們把資金「集中重押」還是「平均分配」。
- 低溫度 (0.5)：極度集中重押 AI 預測第一名
- 高溫度 (5.0)：接近簡單平均等權重分配

我們現在已經確認最佳的持股數量為 40 檔 (TOP_K = 40)。本實驗室將測試在這 40 檔內，不同的資金分配會如何改變風險與報酬。


In [1]:
import os, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import vectorbt as vbt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tqdm import tqdm
import yfinance as yf

# 切換到專案根目錄
original_cwd = os.getcwd()
try:
    if os.path.basename(original_cwd) == "Research":
        os.chdir("..")
        
    print("Loading cached data from finmind_cache...")
    # 1. 載入原始價格與預測資料
    close = pd.read_pickle("finmind_cache/close_wide.pkl")
    close.index = pd.to_datetime(close.index)
    
    money_frames = []
    import glob
    for f in glob.glob("finmind_cache/price/*.pkl"):
        try:
            df = pd.read_pickle(f)
            if "Trading_money" in df.columns:
                tmp = df[["date", "stock_id", "Trading_money"]].copy()
                tmp["Trading_money"] = pd.to_numeric(tmp["Trading_money"], errors="coerce")
                tmp["date"] = pd.to_datetime(tmp["date"])
                money_frames.append(tmp)
        except Exception:
            pass
    
    money_wide = pd.concat(money_frames, ignore_index=True).pivot_table(index="date", columns="stock_id", values="Trading_money")
    money_wide.index = pd.to_datetime(money_wide.index)
    
    # 2. 載入我們策略已經訓練好的預測結果
    print("Loading ML predictions...")
    final_preds = pd.read_pickle("predictions.pkl")
    pred_wide = final_preds["y_pred"].unstack("stock_id")
    
    # 3. 建立多頭過濾網
    tw50 = close["0050"]
    sig_60ma  = (tw50 > tw50.rolling(60).mean()).astype(float) * 0.4
    sig_20ma  = (tw50 > tw50.rolling(20).mean()).astype(float) * 0.3 
    sig_nodip = (tw50.pct_change(1) > -0.05).astype(float) * 0.3    
    regime_score = sig_60ma + sig_20ma + sig_nodip
    is_bull_daily   = regime_score >= 0.4
    is_bull_monthly = (is_bull_daily.resample("ME").mean()
                                    .reindex(pred_wide.index, method="ffill")
                                    .fillna(0) >= 0.4)
    
    # 流動性過濾
    money_m_liq = money_wide.rolling(30).mean().resample("ME").last()
    LIQUIDITY_MIN = 30_000_000
    TEST_START = pd.Timestamp("2020-01-01")
    TOP_K = 40  # 鎖定前一步的最佳化產物
    
    def softmax_weights(scores: pd.Series, temp: float, top_k: int) -> pd.Series:
        top = scores.nlargest(top_k)
        exp_  = np.exp((top - top.max()) / temp)
        return exp_ / exp_.sum()
    
    print(f"Pre-processing done! Scanning temperatures...")
    
    # VectorBT 優化跑迴圈
    sweep_Temps = [0.5, 1.0, 2.0, 3.0, 5.0, 10.0, 20.0]
    results = []
    
    # 準備回測要用的價格資料
    close_clean = (close.loc[TEST_START:]
                   .replace(0, np.nan)
                   .ffill().bfill()
                   .dropna(axis=1))
    
    for temp in sweep_Temps:
        print(f"\nRunning Backtest for WEIGHT_TEMP = {temp} ...")
        weight_rows = {}
        for date, row in pred_wide.iterrows():
            row = row.dropna()
            if date in money_m_liq.index:
                liq_row  = money_m_liq.loc[date].reindex(row.index)
                liq_ok   = liq_row[liq_row >= LIQUIDITY_MIN].index
                row      = row[liq_ok]
            
            if is_bull_monthly.get(date, False) and len(row) >= TOP_K:
                weight_rows[date] = softmax_weights(row, temp, TOP_K)
        
        weights_df = pd.DataFrame.from_dict(weight_rows, orient="index")
        weights_df = weights_df.reindex(pred_wide.index).fillna(0.0)

        weights_daily = weights_df.reindex(close.index, method="ffill").fillna(0.0)
        common      = close_clean.columns.intersection(weights_daily.columns)
        c_vbt       = close_clean[common]
        w_vbt       = weights_daily.loc[TEST_START:, common]
        
        pf = vbt.Portfolio.from_orders(
            close       = c_vbt,
            size        = w_vbt,
            size_type   = "targetpercent",
            group_by    = True,
            cash_sharing= True,
            fees        = 0.001425,
            slippage    = 0.001,
            init_cash   = 1_000_000,
        )
        
        stats = pf.stats()
        total_ret = stats["Total Return [%]"]
        max_dd = stats["Max Drawdown [%]"]
        monthly_ret = pf.value().resample("ME").last().pct_change().dropna()
        sharpe = float((monthly_ret.mean() / monthly_ret.std()) * (12**0.5))
        
        years = (pf.value().index[-1] - pf.value().index[0]).days / 365.25
        cagr = ((pf.value().iloc[-1] / pf.value().iloc[0]) ** (1/years)) - 1
        
        results.append({
            "WEIGHT_TEMP": temp,
            "CAGR": cagr,
            "Sharpe": sharpe,
            "Max DD": max_dd,
            "Total Return": total_ret
        })
        
        print(f"  → CAGR: {cagr:.2%} | Sharpe: {sharpe:.2f} | Max DD: {max_dd:.2f}%")
        
finally:
    os.chdir(original_cwd)


Loading cached data from finmind_cache...
Loading ML predictions...
Pre-processing done! Scanning temperatures...

Running Backtest for WEIGHT_TEMP = 0.5 ...
  → CAGR: 10.00% | Sharpe: 0.55 | Max DD: 34.31%

Running Backtest for WEIGHT_TEMP = 1.0 ...
  → CAGR: 9.99% | Sharpe: 0.55 | Max DD: 34.30%

Running Backtest for WEIGHT_TEMP = 2.0 ...
  → CAGR: 9.98% | Sharpe: 0.55 | Max DD: 34.30%

Running Backtest for WEIGHT_TEMP = 3.0 ...
  → CAGR: 9.98% | Sharpe: 0.55 | Max DD: 34.30%

Running Backtest for WEIGHT_TEMP = 5.0 ...
  → CAGR: 9.98% | Sharpe: 0.55 | Max DD: 34.30%

Running Backtest for WEIGHT_TEMP = 10.0 ...
  → CAGR: 9.97% | Sharpe: 0.55 | Max DD: 34.30%

Running Backtest for WEIGHT_TEMP = 20.0 ...
  → CAGR: 9.97% | Sharpe: 0.55 | Max DD: 34.30%


In [2]:
# ── 整理掃描結果並視覺化 ──
res_df = pd.DataFrame(results).set_index("WEIGHT_TEMP")
print("\n=== Temperature Factor Sweep Results ===")
display(res_df.round(4))

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(
    x=[str(x) for x in res_df.index], 
    y=res_df["CAGR"]*100, 
    name="CAGR (%)",
    marker_color="#00BFA5"
), secondary_y=False)

fig.add_trace(go.Scatter(
    x=[str(x) for x in res_df.index], 
    y=abs(res_df["Max DD"]),
    name="|Max Drawdown| (%)",
    mode="lines+markers",
    line=dict(color="#FF1744", width=3)
), secondary_y=True)

fig.update_layout(
    title="CAGR vs Max Drawdown by Temperature (WEIGHT_TEMP)",
    xaxis_title="Temperature (Low = Concentrated, High = Equal Weights)",
    template="plotly_dark", height=500
)
fig.update_yaxes(title_text="CAGR (%)", secondary_y=False)
fig.update_yaxes(title_text="Max Drawdown (%)", secondary_y=True)

fig.show()


=== Temperature Factor Sweep Results ===


,CAGR,Sharpe,Max DD,Total Return
WEIGHT_TEMP,,,,
0.5,0.1000,0.5530,34.3051,80.0451
1.0,0.0999,0.5524,34.3030,79.8928
2.0,0.0998,0.5520,34.3019,79.8169
3.0,0.0998,0.5519,34.3016,79.7919
5.0,0.0998,0.5518,34.3013,79.7717
10.0,0.0997,0.5517,34.3011,79.7566
20.0,0.0997,0.5517,34.3010,79.7491


In [3]:
# ── 最終結論與建議
top_cagr = res_df["CAGR"].idxmax()
top_sharpe = res_df["Sharpe"].idxmax()

print("PHASE 3-D 結論：")
print(f"1. 最高 CAGR 落在 WEIGHT_TEMP = {top_cagr}")
print(f"2. 最高 Sharpe (CP值) 落在 WEIGHT_TEMP = {top_sharpe}")
print("\n下一步行動：請打開 strategy.py，找到 WEIGHT_TEMP 的宣告區域，")
print("修改為上述的最優解，執行 `python3 strategy.py` 後觀察最終極效能！")

PHASE 3-D 結論：
1. 最高 CAGR 落在 WEIGHT_TEMP = 0.5
2. 最高 Sharpe (CP值) 落在 WEIGHT_TEMP = 0.5

下一步行動：請打開 strategy.py，找到 WEIGHT_TEMP 的宣告區域，
修改為上述的最優解，執行 `python3 strategy.py` 後觀察最終極效能！
